## Hybrid Model — Step 1: Project setup and dataset paths

This cell initializes workspace paths, builds an absolute `data.yaml`, sets reproducibility seeds, and scans train/valid/test image-label counts.

In [4]:
import os
import random
from pathlib import Path

import numpy as np
import torch
import yaml

WORKSPACE = Path(r"D:\xray2")
YOLO_DATA = WORKSPACE / "SIXray.v1i.yolov8"
RAW_DATA_YAML = YOLO_DATA / "data.yaml"
TRAIN_DATA_YAML = YOLO_DATA / "data_abs.yaml"
RUNS_DIR = WORKSPACE / "runs"
ARTIFACTS_DIR = WORKSPACE / "artifacts"

RUNS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


seed_everything(42)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available in this notebook kernel. "
        "Install CUDA-enabled PyTorch in d:/xray2/.venv and restart the kernel."
    )

print("✅ CUDA ready:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda)


def make_absolute_data_yaml(src_yaml: Path, dst_yaml: Path):
    with open(src_yaml, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    names = cfg.get("names", ["Gun", "Knife", "Pliers", "Scissors", "Wrench"])
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names)]

    cfg["path"] = str(YOLO_DATA)
    cfg["train"] = "train/images"
    cfg["val"] = "valid/images"
    cfg["test"] = "test/images"
    cfg["names"] = names
    cfg["nc"] = len(names)

    with open(dst_yaml, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)

    return cfg


cfg = make_absolute_data_yaml(RAW_DATA_YAML, TRAIN_DATA_YAML)


def scan_split(split: str):
    img_dir = YOLO_DATA / split / "images"
    lbl_dir = YOLO_DATA / split / "labels"
    images = [p for p in img_dir.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}]
    labels = list(lbl_dir.glob("*.txt"))
    print(f"{split.upper()} -> Images: {len(images)}, Labels: {len(labels)}")


for split_name in ["train", "valid", "test"]:
    scan_split(split_name)

print("Classes:", cfg["names"])
print("Using data yaml:", TRAIN_DATA_YAML)

✅ CUDA ready: NVIDIA GeForce RTX 3050 OEM
Torch: 2.6.0+cu124 | CUDA: 12.4
TRAIN -> Images: 10536, Labels: 10536
VALID -> Images: 1013, Labels: 1013
TEST -> Images: 541, Labels: 541
Classes: ['Gun', 'Knife', 'Pliers', 'Scissors', 'Wrench']
Using data yaml: D:\xray2\SIXray.v1i.yolov8\data_abs.yaml


## Hybrid Model — Step 2: Dataset cleaning and label consistency

This cell removes invalid image/label pairs, validates YOLO annotation format, and re-checks split counts after cleaning.

In [5]:
VALID_IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp"}


def _label_name_from_image(image_name: str):
    return f"{Path(image_name).stem}.txt"


def clean_split(split: str):
    img_dir = YOLO_DATA / split / "images"
    lbl_dir = YOLO_DATA / split / "labels"

    image_files = [p for p in img_dir.iterdir() if p.is_file() and p.suffix.lower() in VALID_IMAGE_EXT]
    label_files = [p for p in lbl_dir.iterdir() if p.is_file() and p.suffix.lower() == ".txt"]

    image_stems = {p.stem for p in image_files}
    label_stems = {p.stem for p in label_files}

    removed_images = 0
    removed_labels = 0
    removed_invalid_labels = 0

    for image_path in image_files:
        if image_path.stem not in label_stems:
            image_path.unlink(missing_ok=True)
            removed_images += 1

    for label_path in label_files:
        if label_path.stem not in image_stems:
            label_path.unlink(missing_ok=True)
            removed_labels += 1

    for label_path in list(lbl_dir.glob("*.txt")):
        valid = True
        with open(label_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    valid = False
                    break
                cls_id, x, y, w, h = map(float, parts)
                if cls_id < 0 or int(cls_id) >= cfg["nc"]:
                    valid = False
                    break
                if not (0.0 <= x <= 1.0 and 0.0 <= y <= 1.0 and 0.0 < w <= 1.0 and 0.0 < h <= 1.0):
                    valid = False
                    break

        if not valid:
            stem = label_path.stem
            label_path.unlink(missing_ok=True)
            for ext in VALID_IMAGE_EXT:
                img_path = img_dir / f"{stem}{ext}"
                if img_path.exists():
                    img_path.unlink(missing_ok=True)
            removed_invalid_labels += 1

    print(
        f"{split.upper()} cleaned -> "
        f"removed_images: {removed_images}, "
        f"removed_labels: {removed_labels}, "
        f"removed_invalid_pairs: {removed_invalid_labels}"
    )


for split_name in ["train", "valid", "test"]:
    clean_split(split_name)

for split_name in ["train", "valid", "test"]:
    scan_split(split_name)

TRAIN cleaned -> removed_images: 0, removed_labels: 0, removed_invalid_pairs: 0
VALID cleaned -> removed_images: 0, removed_labels: 0, removed_invalid_pairs: 0
TEST cleaned -> removed_images: 0, removed_labels: 0, removed_invalid_pairs: 0
TRAIN -> Images: 10536, Labels: 10536
VALID -> Images: 1013, Labels: 1013
TEST -> Images: 541, Labels: 541


## Hybrid Model — Step 3: YOLO detection training and benchmark

This cell trains YOLO with multiple tuned configs, evaluates on validation/test, and selects the best checkpoint by mAP50-95 then mAP50.

In [6]:
# ==============================================================
# FILE 3 — YOLOv8 TRAINING (2-Stage, Safe + Resume + Retry)
# ==============================================================
import os
import gc
import time
import json
from pathlib import Path

import torch
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Use a CUDA-enabled kernel before running this cell.")

YOLO_BASE = Path(r"D:\xray2\SIXray.v1i.yolov8")
DATA_YAML = YOLO_BASE / "data.yaml"
SAVE_DIR = Path(r"D:\xray2\yolo_runs")
ARTIFACTS_DIR = Path(r"D:\xray2\artifacts")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}


def scan(split: str):
    split_dir = YOLO_BASE / split
    img_dir = split_dir / "images"
    lbl_dir = split_dir / "labels"

    images = [p for p in img_dir.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTS]
    labels = [p for p in lbl_dir.iterdir() if p.is_file() and p.suffix.lower() == ".txt"]
    print(f"  {split.upper()}: {len(images)} images / {len(labels)} labels")


def clean(split: str):
    img_dir = YOLO_BASE / split / "images"
    lbl_dir = YOLO_BASE / split / "labels"

    image_files = [p for p in img_dir.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTS]
    label_files = [p for p in lbl_dir.iterdir() if p.is_file() and p.suffix.lower() == ".txt"]

    image_stems = {p.stem for p in image_files}
    label_stems = {p.stem for p in label_files}

    removed_images = 0
    removed_labels = 0

    for image_path in image_files:
        if image_path.stem not in label_stems:
            image_path.unlink(missing_ok=True)
            removed_images += 1

    for label_path in label_files:
        if label_path.stem not in image_stems:
            label_path.unlink(missing_ok=True)
            removed_labels += 1

    print(f"  {split}: removed_images={removed_images}, removed_labels={removed_labels}")


def clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def train_with_recovery(stage_name: str, base_model_path: str, train_kwargs: dict, run_name: str):
    run_dir = SAVE_DIR / run_name
    last_path = run_dir / "weights" / "last.pt"
    best_path = run_dir / "weights" / "best.pt"

    attempts = [
        {"workers": 0, "cache": "disk", "multi_scale": False},
        {"workers": 0, "cache": False, "multi_scale": False, "mosaic": min(float(train_kwargs.get("mosaic", 0.0)), 0.5), "mixup": min(float(train_kwargs.get("mixup", 0.0)), 0.05)},
        {"workers": 0, "cache": False, "multi_scale": False, "mosaic": 0.0, "mixup": 0.0},
    ]

    last_error = None

    for attempt_idx, fallback in enumerate(attempts, start=1):
        clear_cuda()
        print(f"\n[{stage_name}] Attempt {attempt_idx}/{len(attempts)} with fallback={fallback}")

        resume_mode = last_path.exists()
        model_path = str(last_path) if resume_mode else base_model_path

        current_kwargs = dict(train_kwargs)
        current_kwargs.update({
            "workers": fallback.get("workers", current_kwargs.get("workers", 0)),
            "cache": fallback.get("cache", current_kwargs.get("cache", "disk")),
            "multi_scale": fallback.get("multi_scale", current_kwargs.get("multi_scale", False)),
            "mosaic": fallback.get("mosaic", current_kwargs.get("mosaic", 0.0)),
            "mixup": fallback.get("mixup", current_kwargs.get("mixup", 0.0)),
        })

        if resume_mode:
            current_kwargs["resume"] = True

        try:
            model = YOLO(model_path)
            model.train(**current_kwargs)

            if best_path.exists():
                return str(best_path)
            raise FileNotFoundError(f"{stage_name}: best.pt not found at {best_path}")

        except RuntimeError as error:
            last_error = error
            message = str(error).lower()
            retryable = (
                "input and output sizes should be greater than 0" in message
                or "out of memory" in message
                or "cudnn" in message
                or "cuda" in message
            )
            print(f"[{stage_name}] RuntimeError: {error}")
            if (not retryable) or attempt_idx == len(attempts):
                raise
            print(f"[{stage_name}] Retrying with safer settings...")
            time.sleep(2)
        except Exception as error:
            last_error = error
            print(f"[{stage_name}] Non-retryable error: {error}")
            raise

    if last_error is not None:
        raise last_error
    raise RuntimeError(f"{stage_name}: training failed unexpectedly")


print("=== Dataset Scan ===")
for split_name in ["train", "valid", "test"]:
    scan(split_name)

print("\n=== Cleaning ===")
for split_name in ["train", "valid", "test"]:
    clean(split_name)

print("\n=== Post-clean ===")
for split_name in ["train", "valid", "test"]:
    scan(split_name)

# ---------------- Stage 1 ----------------
print("\n=== Stage 1: YOLOv8m — 80 epochs @ 640px ===")

stage1_kwargs = dict(
    data=str(DATA_YAML),
    project=str(SAVE_DIR),
    name="s1_yolov8m",
    epochs=80,
    imgsz=640,
    batch=8,
    device=0,
    optimizer="AdamW",
    lr0=5e-4,
    lrf=0.01,
    weight_decay=5e-4,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    flipud=0.1,
    fliplr=0.5,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.0005,
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.4,
    patience=25,
    warmup_epochs=5,
    close_mosaic=10,
    workers=0,
    cache="disk",
    save=True,
    save_period=10,
    val=True,
    plots=True,
    verbose=True,
    exist_ok=True,
    pretrained=True,
    multi_scale=False,
)

stage1_best = train_with_recovery(
    stage_name="Stage 1",
    base_model_path="yolov8m.pt",
    train_kwargs=stage1_kwargs,
    run_name="s1_yolov8m",
)
print(f"\n✅ Stage 1 done -> {stage1_best}")

if not Path(stage1_best).exists():
    raise FileNotFoundError(f"Stage-1 best weights missing: {stage1_best}")

# ---------------- Stage 2 ----------------
print("\n=== Stage 2: Fine-tune @ 832px — 30 epochs ===")

stage2_kwargs = dict(
    data=str(DATA_YAML),
    project=str(SAVE_DIR),
    name="s2_finetune",
    epochs=30,
    imgsz=832,
    batch=4,
    device=0,
    optimizer="AdamW",
    lr0=5e-5,
    lrf=0.1,
    weight_decay=5e-4,
    mosaic=0.5,
    mixup=0.05,
    copy_paste=0.0,
    flipud=0.05,
    fliplr=0.5,
    degrees=5.0,
    translate=0.05,
    scale=0.3,
    shear=0.0,
    perspective=0.0,
    hsv_h=0.01,
    hsv_s=0.2,
    hsv_v=0.2,
    patience=15,
    warmup_epochs=2,
    close_mosaic=5,
    workers=0,
    cache="disk",
    save=True,
    val=True,
    plots=True,
    verbose=True,
    exist_ok=True,
    pretrained=True,
    multi_scale=False,
)

stage2_best = train_with_recovery(
    stage_name="Stage 2",
    base_model_path=stage1_best,
    train_kwargs=stage2_kwargs,
    run_name="s2_finetune",
)
print(f"\n✅ Stage 2 fine-tune done -> {stage2_best}")

if not Path(stage2_best).exists():
    raise FileNotFoundError(f"Stage-2 best weights missing: {stage2_best}")

# ---------------- Test Evaluation ----------------
print("\n=== Test Set Evaluation ===")
final_model = YOLO(stage2_best)
metrics = final_model.val(data=str(DATA_YAML), split="test", device=0, imgsz=832, verbose=True)
print(f"\nmAP@50:    {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")

best_meta = {
    "weights": str(stage2_best),
    "map50": float(metrics.box.map50),
    "map5095": float(metrics.box.map),
    "source": "Step 3 two-stage YOLO",
}
best_meta_path = ARTIFACTS_DIR / "best_yolo_hybrid.json"
with open(best_meta_path, "w", encoding="utf-8") as f:
    json.dump(best_meta, f, indent=2)
print(f"✅ Saved YOLO metadata: {best_meta_path}")

=== Dataset Scan ===
  TRAIN: 10536 images / 10536 labels
  VALID: 1013 images / 1013 labels
  TEST: 541 images / 541 labels

=== Cleaning ===
  train: removed_images=0, removed_labels=0
  valid: removed_images=0, removed_labels=0
  test: removed_images=0, removed_labels=0

=== Post-clean ===
  TRAIN: 10536 images / 10536 labels
  VALID: 1013 images / 1013 labels
  TEST: 541 images / 541 labels

=== Stage 1: YOLOv8m — 80 epochs @ 640px ===

[Stage 1] Attempt 1/3 with fallback={'workers': 0, 'cache': 'disk', 'multi_scale': False}
Ultralytics 8.4.17  Python-3.10.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 OEM, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\xray2\SIXray.v1i.yolov8\data.yaml, degrees=10.0, deterministic=True, device=0, df

## Hybrid Model — Step 4: SSL pretraining (SimCLR + ViT-Small)

This cell learns contrastive ROI representations for weapon semantics and saves the SSL backbone weights for downstream prototype modeling.

In [7]:
# ==============================================================
# FILE 1 — SSL PRE-TRAINING  (SimCLR  ·  ViT-Small)
# ==============================================================
# KEY FIXES vs your original:
#   ✅ Crops bbox regions → model learns objects, not background
#   ✅ Two DIFFERENT augmentation views (was same transform twice)
#   ✅ Lower temperature 0.07 (was 0.5) → harder negatives
#   ✅ 3-layer projector with GELU (was 2-layer ReLU)
#   ✅ Linear warmup + cosine (was plain cosine)
#   ✅ Larger projection dim 256 (was 128)
#   ✅ Both datasets: COCO + YOLOv8
# ==============================================================

import os, time
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import timm

# ─────────────────────────── CONFIG ───────────────────────────
WORKSPACE = Path(r"D:\xray2")
ARTIFACTS_DIR = WORKSPACE / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
YOLO_TRAIN_IMAGES = str(WORKSPACE / "SIXray.v1i.yolov8" / "train" / "images")
YOLO_TRAIN_LABELS = str(WORKSPACE / "SIXray.v1i.yolov8" / "train" / "labels")

SAVE_PATH       = ARTIFACTS_DIR / "ssl_vit_small_backbone.pth"

IMG_SIZE        = 224
BATCH_SIZE      = 32           # was 16 — better NT-Xent gradients
EPOCHS          = 50           # was 30
LR              = 3e-4
WARMUP_EPOCHS   = 5
TEMPERATURE     = 0.07         # was 0.5  (critical fix — harder negatives)
PROJECTION_DIM  = 256          # was 128
NUM_WORKERS     = 0            # keep 0 on Windows

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ──────────────────── TWO DISTINCT VIEWS ─────────────────────
# FIX: Your original code applied the SAME transform to x1 & x2
# which means both views are nearly identical → no contrastive signal
# View-1 = strong spatial + color  |  View-2 = lighter / different

view1 = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.4, 1.0)),   # aggressive crop
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(15),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))
    ], p=0.4),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1)
    ], p=0.5),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

view2 = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),   # gentler crop
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.1, contrast=0.1)
    ], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ─────────────── DATASET  (bbox crops, not whole images) ──────
# FIX: Original used whole images. Cropping bboxes forces the
# SSL backbone to learn discriminative object features, which
# is exactly what the prototype stage needs later.

class XrayCropSSLDataset(Dataset):
    def __init__(self, img_dir, lbl_dir):
        self.samples = []
        for fname in os.listdir(img_dir):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            img_path = os.path.join(img_dir, fname)
            lbl_name = (fname.replace(".jpg", ".txt")
                            .replace(".jpeg", ".txt")
                            .replace(".png", ".txt"))
            lbl_path = os.path.join(lbl_dir, lbl_name)
            if os.path.exists(lbl_path):
                with open(lbl_path) as f:
                    lines = [l.strip() for l in f if l.strip()]
                for line in lines:
                    parts = line.split()
                    if len(parts) == 5:
                        _, cx, cy, bw, bh = map(float, parts)
                        self.samples.append((img_path, (cx, cy, bw, bh)))
            else:
                self.samples.append((img_path, None))   # fallback: whole image

        print(f"SSL Dataset: {len(self.samples)} crops/images")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, bbox = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        W, H = img.size

        if bbox is not None:
            cx, cy, bw, bh = bbox
            x1 = max(0, int((cx - bw / 2) * W))
            y1 = max(0, int((cy - bh / 2) * H))
            x2 = min(W, int((cx + bw / 2) * W))
            y2 = min(H, int((cy + bh / 2) * H))
            if x2 > x1 + 4 and y2 > y1 + 4:
                img = img.crop((x1, y1, x2, y2))

        return view1(img), view2(img)   # two DIFFERENT views


dataset = XrayCropSSLDataset(YOLO_TRAIN_IMAGES, YOLO_TRAIN_LABELS)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                     num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
print(f"Batches/epoch: {len(loader)}")

# ──────────────────────── MODEL ───────────────────────────────
class SimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            "vit_small_patch16_224", pretrained=True, num_classes=0)
        dim = self.backbone.num_features   # 384

        # FIX: 3-layer MLP projector (was 2-layer) — standard best practice
        self.projector = nn.Sequential(
            nn.Linear(dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Linear(512, PROJECTION_DIM),
            nn.BatchNorm1d(PROJECTION_DIM),
        )

    def forward(self, x):
        return self.projector(self.backbone(x))


model     = SimCLR().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR,
                               weight_decay=1e-4, betas=(0.9, 0.95))

# FIX: Linear warmup then cosine (was plain cosine from epoch 0)
def lr_lambda(ep):
    if ep < WARMUP_EPOCHS:
        return (ep + 1) / WARMUP_EPOCHS
    prog = (ep - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    import math
    return 0.5 * (1.0 + math.cos(math.pi * prog))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler    = torch.amp.GradScaler("cuda")

# ─────────────────── NT-XENT LOSS ─────────────────────────────
def nt_xent_loss(z1, z2):
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    N  = z1.size(0)
    z  = torch.cat([z1, z2], dim=0)                      # (2N, D)
    sim = torch.matmul(z, z.T) / TEMPERATURE              # (2N, 2N)
    sim.fill_diagonal_(float("-inf"))                     # remove self-sim
    # positives: i↔i+N  and  i+N↔i
    pos = torch.cat([torch.diag(sim, N), torch.diag(sim, -N)])  # (2N,)
    loss = -pos + torch.logsumexp(sim, dim=1)
    return loss.mean()

# ──────────────────── TRAINING LOOP ───────────────────────────
best_loss = float("inf")
print("\nStarting SSL Pre-Training…\n")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    t0 = time.time()

    for x1, x2 in loader:
        x1 = x1.to(DEVICE, non_blocking=True)
        x2 = x2.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            loss = nt_xent_loss(model(x1), model(x2))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    scheduler.step()
    avg  = total_loss / len(loader)
    lr_  = optimizer.param_groups[0]["lr"]
    mins = (time.time() - t0) / 60
    print(f"Epoch [{epoch+1:3d}/{EPOCHS}] Loss: {avg:.4f} | LR: {lr_:.2e} | {mins:.1f}min")

    if avg < best_loss:
        best_loss = avg
        torch.save(model.backbone.state_dict(), SAVE_PATH)
        print(f"  ✅ Best backbone saved → {SAVE_PATH}")

# Collapse check
model.eval()
with torch.no_grad():
    x1, _ = next(iter(loader))
    feats  = model.backbone(x1.to(DEVICE))
    print(f"\nFeature variance (>0.01 = healthy): {feats.var().item():.4f}")

print(f"\n✅ SSL done. Backbone saved: {SAVE_PATH}")


Device: cuda
GPU: NVIDIA GeForce RTX 3050 OEM
SSL Dataset: 19864 crops/images
Batches/epoch: 620

Starting SSL Pre-Training…

Epoch [  1/50] Loss: 0.0463 | LR: 1.20e-04 | 8.0min
  ✅ Best backbone saved → D:\xray2\artifacts\ssl_vit_small_backbone.pth
Epoch [  2/50] Loss: 0.0236 | LR: 1.80e-04 | 5.8min
  ✅ Best backbone saved → D:\xray2\artifacts\ssl_vit_small_backbone.pth
Epoch [  3/50] Loss: 0.0315 | LR: 2.40e-04 | 5.9min
Epoch [  4/50] Loss: 0.0469 | LR: 3.00e-04 | 5.7min
Epoch [  5/50] Loss: 0.0605 | LR: 3.00e-04 | 5.8min
Epoch [  6/50] Loss: 0.0549 | LR: 3.00e-04 | 5.7min
Epoch [  7/50] Loss: 0.0479 | LR: 2.99e-04 | 5.8min
Epoch [  8/50] Loss: 0.0423 | LR: 2.97e-04 | 5.8min
Epoch [  9/50] Loss: 0.0395 | LR: 2.94e-04 | 5.8min
Epoch [ 10/50] Loss: 0.0385 | LR: 2.91e-04 | 5.7min
Epoch [ 11/50] Loss: 0.0354 | LR: 2.87e-04 | 5.8min
Epoch [ 12/50] Loss: 0.0318 | LR: 2.82e-04 | 5.6min
Epoch [ 13/50] Loss: 0.0302 | LR: 2.77e-04 | 5.7min
Epoch [ 14/50] Loss: 0.0285 | LR: 2.71e-04 | 5.6min
Ep

## Hybrid Model — Step 5: Multi-prototype generation with K-Means

This cell extracts SSL features per class from labeled ROIs and builds multiple class prototypes (`K=3`) for robust similarity matching.

In [8]:
import os

import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import timm
import numpy as np
from sklearn.cluster import KMeans
from tqdm import tqdm

# ================= CONFIG =================
PROTO_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMAGES_DIR = YOLO_DATA / "train" / "images"
LABELS_DIR = YOLO_DATA / "train" / "labels"

SSL_WEIGHTS = ARTIFACTS_DIR / "ssl_vit_small_backbone.pth"
PROTOTYPE_PATH = ARTIFACTS_DIR / "prototypes_multi_v1.pth"

CLASS_NAMES = cfg["names"]
K_CLUSTERS = 3
IMG_SIZE = 224
MAX_SAMPLES_PER_CLASS = 6000

print("Using device:", PROTO_DEVICE)
print("Loading SSL backbone weights from:", SSL_WEIGHTS)

# ================= LOAD SSL BACKBONE =================
model = timm.create_model("vit_small_patch16_224", pretrained=False, num_classes=0).to(PROTO_DEVICE)
model.load_state_dict(torch.load(SSL_WEIGHTS, map_location=PROTO_DEVICE))
model.eval()

# ================= TRANSFORM =================
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

@torch.no_grad()
def extract_feature(img):
    x = transform(img).unsqueeze(0).to(PROTO_DEVICE)
    feat = model(x)
    return F.normalize(feat, dim=1).cpu().numpy().reshape(-1)

# ================= COLLECT FEATURES =================
features_per_class = {class_name: [] for class_name in CLASS_NAMES}

for img_name in tqdm(os.listdir(IMAGES_DIR)):
    if not img_name.lower().endswith((".jpg", ".png", ".jpeg", ".bmp")):
        continue

    img_path = IMAGES_DIR / img_name
    label_path = LABELS_DIR / f"{os.path.splitext(img_name)[0]}.txt"

    if not label_path.exists():
        continue

    img = Image.open(img_path).convert("RGB")
    image_width, image_height = img.size

    with open(label_path, "r", encoding="utf-8") as file:
        lines = file.readlines()

    for line in lines:
        class_id, x_center, y_center, box_width, box_height = map(float, line.strip().split())
        class_id = int(class_id)

        if class_id < 0 or class_id >= len(CLASS_NAMES):
            continue

        x1 = max(0, int((x_center - box_width / 2) * image_width))
        y1 = max(0, int((y_center - box_height / 2) * image_height))
        x2 = min(image_width, int((x_center + box_width / 2) * image_width))
        y2 = min(image_height, int((y_center + box_height / 2) * image_height))

        if x2 <= x1 or y2 <= y1:
            continue

        crop = img.crop((x1, y1, x2, y2))
        feat = extract_feature(crop)
        features_per_class[CLASS_NAMES[class_id]].append(feat)

# ================= CLUSTERING =================
prototypes = {}

for class_name in CLASS_NAMES:
    class_feats = features_per_class[class_name]

    if len(class_feats) < K_CLUSTERS:
        raise ValueError(f"Not enough samples for class {class_name}: {len(class_feats)}")

    if len(class_feats) > MAX_SAMPLES_PER_CLASS:
        selected_indices = np.random.choice(len(class_feats), MAX_SAMPLES_PER_CLASS, replace=False)
        class_feats = [class_feats[idx] for idx in selected_indices]

    feats = np.stack(class_feats, axis=0)
    print(f"Clustering {class_name} -> {len(feats)} samples")

    kmeans = KMeans(n_clusters=K_CLUSTERS, random_state=42, n_init=20)
    kmeans.fit(feats)

    centroids = torch.tensor(kmeans.cluster_centers_, dtype=torch.float32)
    centroids = F.normalize(centroids, dim=1)
    prototypes[class_name] = centroids

# ================= SAVE =================
torch.save(prototypes, PROTOTYPE_PATH)
print(f"\n✅ Multi-prototypes saved to: {PROTOTYPE_PATH}")

Using device: cuda
Loading SSL backbone weights from: D:\xray2\artifacts\ssl_vit_small_backbone.pth


100%|██████████| 21072/21072 [04:18<00:00, 81.49it/s] 


Clustering Gun -> 5484 samples
Clustering Knife -> 2678 samples
Clustering Pliers -> 6000 samples
Clustering Scissors -> 1402 samples
Clustering Wrench -> 3641 samples

✅ Multi-prototypes saved to: D:\xray2\artifacts\prototypes_multi_v1.pth


## Hybrid Model — Step 6: Class-adaptive threshold estimation

This cell computes per-class similarity score distributions and saves adaptive thresholds used to gate weak or noisy detections.

In [9]:
import os

import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import timm
import numpy as np
from tqdm import tqdm

# ================= CONFIG =================
THR_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMAGES_DIR = YOLO_DATA / "train" / "images"
LABELS_DIR = YOLO_DATA / "train" / "labels"

SSL_WEIGHTS = ARTIFACTS_DIR / "ssl_vit_small_backbone.pth"
PROTOTYPE_PATH = ARTIFACTS_DIR / "prototypes_multi_v1.pth"
THRESHOLD_PATH = ARTIFACTS_DIR / "adaptive_thresholds.pth"

CLASS_NAMES = cfg["names"]
IMG_SIZE = 224

print("Using device:", THR_DEVICE)

# ================= LOAD SSL BACKBONE =================
model = timm.create_model("vit_small_patch16_224", pretrained=False, num_classes=0).to(THR_DEVICE)
model.load_state_dict(torch.load(SSL_WEIGHTS, map_location=THR_DEVICE))
model.eval()

# ================= LOAD PROTOTYPES =================
prototypes = torch.load(PROTOTYPE_PATH, map_location=THR_DEVICE)
for key in prototypes:
    prototypes[key] = F.normalize(prototypes[key].to(THR_DEVICE), dim=1)

# ================= TRANSFORM =================
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

@torch.no_grad()
def extract_feature(img):
    x = transform(img).unsqueeze(0).to(THR_DEVICE)
    feat = model(x)
    return F.normalize(feat, dim=1)

# ================= COLLECT SIMILARITY DISTRIBUTIONS =================
similarity_scores = {class_name: [] for class_name in CLASS_NAMES}

for img_name in tqdm(os.listdir(IMAGES_DIR)):
    if not img_name.lower().endswith((".jpg", ".png", ".jpeg", ".bmp")):
        continue

    img_path = IMAGES_DIR / img_name
    label_path = LABELS_DIR / f"{os.path.splitext(img_name)[0]}.txt"

    if not label_path.exists():
        continue

    img = Image.open(img_path).convert("RGB")
    image_width, image_height = img.size

    with open(label_path, "r", encoding="utf-8") as file:
        lines = file.readlines()

    for line in lines:
        class_id, x_center, y_center, box_width, box_height = map(float, line.strip().split())
        class_id = int(class_id)

        if class_id < 0 or class_id >= len(CLASS_NAMES):
            continue

        x1 = max(0, int((x_center - box_width / 2) * image_width))
        y1 = max(0, int((y_center - box_height / 2) * image_height))
        x2 = min(image_width, int((x_center + box_width / 2) * image_width))
        y2 = min(image_height, int((y_center + box_height / 2) * image_height))

        if x2 <= x1 or y2 <= y1:
            continue

        crop = img.crop((x1, y1, x2, y2))
        feat = extract_feature(crop)

        sims = F.cosine_similarity(feat, prototypes[CLASS_NAMES[class_id]])
        similarity_scores[CLASS_NAMES[class_id]].append(float(sims.max().item()))

# ================= COMPUTE ADAPTIVE THRESHOLDS =================
adaptive_thresholds = {}

for class_name in CLASS_NAMES:
    scores = np.array(similarity_scores[class_name], dtype=np.float32)

    if len(scores) == 0:
        print(f"⚠ WARNING: No samples found for {class_name}")
        adaptive_thresholds[class_name] = 0.35
        continue

    raw_threshold = float(np.percentile(scores, 10))
    clipped_threshold = float(np.clip(raw_threshold, 0.30, 0.85))
    adaptive_thresholds[class_name] = clipped_threshold

    print(f"{class_name} threshold = {clipped_threshold:.4f} (raw p10={raw_threshold:.4f})")

# ================= SAVE =================
torch.save(adaptive_thresholds, THRESHOLD_PATH)
print(f"\n✅ Adaptive thresholds saved to: {THRESHOLD_PATH}")

Using device: cuda


100%|██████████| 21072/21072 [03:20<00:00, 105.06it/s]

Gun threshold = 0.8500 (raw p10=0.9341)
Knife threshold = 0.8500 (raw p10=0.9304)
Pliers threshold = 0.8500 (raw p10=0.9385)
Scissors threshold = 0.8500 (raw p10=0.9484)
Wrench threshold = 0.8500 (raw p10=0.9343)

✅ Adaptive thresholds saved to: D:\xray2\artifacts\adaptive_thresholds.pth


## Hybrid Model — Step 7: Final hybrid fusion evaluation

This cell runs YOLO inference, verifies detections via SSL prototype similarity, applies confidence fusion (`0.75 YOLO + 0.25 SSL`), and computes final AP/mAP metrics.

In [8]:
import json
import os
from pathlib import Path

import torch
import torch.nn.functional as F
from ultralytics import YOLO
from torchvision import transforms
from PIL import Image
import timm
import numpy as np
import yaml

# ================= CONFIG =================
candidate_roots = [Path.cwd(), Path(r"D:\xray2\xray2"), Path(r"D:\xray2")]
WORKSPACE = None
for root in candidate_roots:
    if (root / "SIXray.v1i.yolov8").exists():
        WORKSPACE = root
        break
if WORKSPACE is None:
    raise FileNotFoundError("Could not locate SIXray.v1i.yolov8 under expected workspace roots.")

YOLO_DATA = WORKSPACE / "SIXray.v1i.yolov8"
ARTIFACTS_DIR = WORKSPACE / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

BASE_PATH = YOLO_DATA
TEST_IMAGES = BASE_PATH / "test" / "images"
TEST_LABELS = BASE_PATH / "test" / "labels"

BEST_YOLO_META = ARTIFACTS_DIR / "best_yolo_hybrid.json"
SSL_WEIGHTS = ARTIFACTS_DIR / "ssl_vit_small_backbone.pth"
PROTOTYPE_PATH = ARTIFACTS_DIR / "prototypes_multi_v1.pth"
THRESHOLD_PATH = ARTIFACTS_DIR / "adaptive_thresholds.pth"

EVAL_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

YOLO_MIN_CONF = 0.05
ALPHA = 0.75
BETA = 0.25
FUSION_THRESHOLD = 0.45

if 'cfg' in globals() and 'names' in cfg:
    CLASS_NAMES = cfg["names"]
else:
    with open(BASE_PATH / "data.yaml", "r", encoding="utf-8") as f:
        data_cfg = yaml.safe_load(f)
    names = data_cfg.get("names", ["Gun", "Knife", "Pliers", "Scissors", "Wrench"])
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names)]
    CLASS_NAMES = names

NUM_CLASSES = len(CLASS_NAMES)

# ================= LOAD MODELS =================
if not BEST_YOLO_META.exists():
    raise FileNotFoundError(f"Run Hybrid Step 3 first. Missing: {BEST_YOLO_META}")

with open(BEST_YOLO_META, "r", encoding="utf-8") as file:
    best_info = json.load(file)

YOLO_WEIGHTS = Path(best_info["weights"])
if not YOLO_WEIGHTS.exists():
    alt_paths = [
        WORKSPACE / "yolo_runs" / "s2_finetune" / "weights" / "best.pt",
        WORKSPACE / YOLO_WEIGHTS.name,
        WORKSPACE / "artifacts" / YOLO_WEIGHTS.name,
    ]
    found = next((p for p in alt_paths if p.exists()), None)
    if found is None:
        raise FileNotFoundError(f"Could not find YOLO weights. Checked: {[str(p) for p in alt_paths]} and metadata path {YOLO_WEIGHTS}")
    YOLO_WEIGHTS = found

print("Loading YOLO weights:", YOLO_WEIGHTS)
yolo = YOLO(str(YOLO_WEIGHTS))

print("Loading SSL backbone...")
ssl_model = timm.create_model("vit_small_patch16_224", pretrained=False, num_classes=0).to(EVAL_DEVICE)
ssl_model.load_state_dict(torch.load(SSL_WEIGHTS, map_location=EVAL_DEVICE))
ssl_model.eval()

print("Loading prototypes and thresholds...")
prototypes = torch.load(PROTOTYPE_PATH, map_location=EVAL_DEVICE)
for key in prototypes:
    prototypes[key] = F.normalize(prototypes[key].to(EVAL_DEVICE), dim=1)
thresholds = torch.load(THRESHOLD_PATH)

# ================= TRANSFORM =================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

@torch.no_grad()
def extract_feature(img):
    x = transform(img).unsqueeze(0).to(EVAL_DEVICE)
    feat = ssl_model(x)
    return F.normalize(feat, dim=1)


def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)
    if inter <= 0:
        return 0.0

    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])
    return inter / (area1 + area2 - inter + 1e-6)


def compute_ap(detections, annotations, class_id, iou_threshold):
    class_dets = [det for det in detections if det["class"] == class_id]
    class_gts = [gt for gt in annotations if gt["class"] == class_id]

    if len(class_gts) == 0:
        return 0.0

    class_dets = sorted(class_dets, key=lambda item: item["score"], reverse=True)

    true_positives = np.zeros(len(class_dets), dtype=np.float32)
    false_positives = np.zeros(len(class_dets), dtype=np.float32)
    matched = set()

    for index, det in enumerate(class_dets):
        gt_img = [gt for gt in class_gts if gt["image"] == det["image"]]

        best_iou = 0.0
        best_gt_idx = -1

        for gt_index, gt in enumerate(gt_img):
            iou = compute_iou(det["box"], gt["box"])
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_index

        match_key = (det["image"], best_gt_idx)
        if best_iou >= iou_threshold and best_gt_idx >= 0 and match_key not in matched:
            true_positives[index] = 1
            matched.add(match_key)
        else:
            false_positives[index] = 1

    cum_tp = np.cumsum(true_positives)
    cum_fp = np.cumsum(false_positives)

    recalls = cum_tp / (len(class_gts) + 1e-6)
    precisions = cum_tp / (cum_tp + cum_fp + 1e-6)

    recalls = np.concatenate(([0.0], recalls, [1.0]))
    precisions = np.concatenate(([1.0], precisions, [0.0]))

    for index in range(len(precisions) - 2, -1, -1):
        precisions[index] = max(precisions[index], precisions[index + 1])

    return float(np.trapezoid(precisions, recalls))

# ================= STORAGE =================
all_detections = []
all_annotations = []

print("Starting Hybrid Evaluation...")

# ================= EVALUATION LOOP =================
for img_name in os.listdir(TEST_IMAGES):
    if not img_name.lower().endswith((".jpg", ".png", ".jpeg", ".bmp")):
        continue

    img_path = TEST_IMAGES / img_name
    label_path = TEST_LABELS / f"{os.path.splitext(img_name)[0]}.txt"

    img = Image.open(img_path).convert("RGB")
    width, height = img.size

    if label_path.exists():
        with open(label_path, "r", encoding="utf-8") as file:
            for line in file:
                cls, x, y, bw, bh = map(float, line.strip().split())
                cls = int(cls)

                x1 = max(0, int((x - bw / 2) * width))
                y1 = max(0, int((y - bh / 2) * height))
                x2 = min(width, int((x + bw / 2) * width))
                y2 = min(height, int((y + bh / 2) * height))

                if x2 <= x1 or y2 <= y1:
                    continue

                all_annotations.append({"image": img_name, "class": cls, "box": [x1, y1, x2, y2]})

    yolo_result = yolo.predict(str(img_path), conf=YOLO_MIN_CONF, device=0 if EVAL_DEVICE == "cuda" else "cpu", verbose=False)[0]

    if yolo_result.boxes is None or len(yolo_result.boxes) == 0:
        continue

    for box, cls_id, conf in zip(yolo_result.boxes.xyxy, yolo_result.boxes.cls, yolo_result.boxes.conf):
        cls_id = int(cls_id)
        if cls_id < 0 or cls_id >= len(CLASS_NAMES):
            continue

        x1, y1, x2, y2 = map(int, box.tolist())
        x1 = max(0, min(width - 1, x1))
        y1 = max(0, min(height - 1, y1))
        x2 = max(0, min(width, x2))
        y2 = max(0, min(height, y2))

        if x2 <= x1 or y2 <= y1:
            continue

        crop = img.crop((x1, y1, x2, y2))
        feat = extract_feature(crop)

        sims = F.cosine_similarity(feat, prototypes[CLASS_NAMES[cls_id]])
        ssl_score = float((sims.max().item() + 1.0) / 2.0)

        if ssl_score < thresholds[CLASS_NAMES[cls_id]]:
            continue

        final_score = ALPHA * float(conf) + BETA * ssl_score
        if final_score < FUSION_THRESHOLD:
            continue

        all_detections.append(
            {
                "image": img_name,
                "class": cls_id,
                "score": final_score,
                "box": [x1, y1, x2, y2],
            }
        )

# Build class labels needed by the next metrics cell.
# Background class index is NUM_CLASSES.
all_gt_labels = []
all_pred_labels = []
for img_name in sorted({item["image"] for item in all_annotations} | {item["image"] for item in all_detections}):
    gts = [g for g in all_annotations if g["image"] == img_name]
    preds = sorted([d for d in all_detections if d["image"] == img_name], key=lambda x: x["score"], reverse=True)

    used_gt = set()
    for pred in preds:
        best_iou = 0.0
        best_gt_idx = -1
        for gt_idx, gt in enumerate(gts):
            if gt_idx in used_gt:
                continue
            iou = compute_iou(pred["box"], gt["box"])
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx

        if best_gt_idx >= 0 and best_iou >= 0.5:
            used_gt.add(best_gt_idx)
            all_gt_labels.append(gts[best_gt_idx]["class"])
            all_pred_labels.append(pred["class"])
        else:
            all_gt_labels.append(NUM_CLASSES)
            all_pred_labels.append(pred["class"])

    for gt_idx, gt in enumerate(gts):
        if gt_idx not in used_gt:
            all_gt_labels.append(gt["class"])
            all_pred_labels.append(NUM_CLASSES)

# ================= FINAL RESULTS =================
ious = np.arange(0.50, 1.00, 0.05)
map_per_iou = []

for iou_thr in ious:
    aps = [compute_ap(all_detections, all_annotations, class_id, iou_thr) for class_id in range(len(CLASS_NAMES))]
    map_iou = float(np.mean(aps)) if len(aps) else 0.0
    map_per_iou.append(map_iou)

map50 = map_per_iou[0] if len(map_per_iou) else 0.0
map5095 = float(np.mean(map_per_iou)) if len(map_per_iou) else 0.0

print("\n===== HYBRID RESULTS =====")
for class_id, class_name in enumerate(CLASS_NAMES):
    class_ap50 = compute_ap(all_detections, all_annotations, class_id, 0.50)
    print(f"{class_name} AP@0.50: {class_ap50:.4f}")

print("\nmAP@0.50:", round(map50, 4))
print("mAP@0.50:0.95:", round(map5095, 4))
print("\nTarget check:")
print("mAP50 >= 0.90:", map50 >= 0.90)
print("mAP50-95 >= 0.70:", map5095 >= 0.70)
print(f"Matched label pairs for precision/recall cell: {len(all_gt_labels)}")

Loading YOLO weights: d:\xray2\xray2\yolo_runs\s2_finetune\weights\best.pt
Loading SSL backbone...
Loading prototypes and thresholds...
Starting Hybrid Evaluation...

===== HYBRID RESULTS =====
Gun AP@0.50: 0.9837
Knife AP@0.50: 0.8882
Pliers AP@0.50: 0.9436
Scissors AP@0.50: 0.9366
Wrench AP@0.50: 0.9067

mAP@0.50: 0.9318
mAP@0.50:0.95: 0.7093

Target check:
mAP50 >= 0.90: True
mAP50-95 >= 0.70: True
Matched label pairs for precision/recall cell: 1101


In [9]:
# Overall precision and recall for Hybrid model (object classes only)
import numpy as np

if 'all_gt_labels' not in globals() or 'all_pred_labels' not in globals() or 'NUM_CLASSES' not in globals():
    print('Run Cell 14 first to generate all_gt_labels/all_pred_labels.')
else:
    y_true = np.array(all_gt_labels)
    y_pred = np.array(all_pred_labels)

    # Foreground classes are 0..NUM_CLASSES-1; NUM_CLASSES is background
    tp = int(np.sum((y_true < NUM_CLASSES) & (y_pred < NUM_CLASSES) & (y_true == y_pred)))
    fp = int(np.sum((y_true == NUM_CLASSES) & (y_pred < NUM_CLASSES))) + int(np.sum((y_true < NUM_CLASSES) & (y_pred < NUM_CLASSES) & (y_true != y_pred)))
    fn = int(np.sum((y_true < NUM_CLASSES) & (y_pred == NUM_CLASSES))) + int(np.sum((y_true < NUM_CLASSES) & (y_pred < NUM_CLASSES) & (y_true != y_pred)))

    overall_precision = tp / (tp + fp + 1e-9)
    overall_recall = tp / (tp + fn + 1e-9)

    print('=== Hybrid Overall Metrics ===')
    print(f'TP: {tp}, FP: {fp}, FN: {fn}')
    print(f'Overall Precision: {overall_precision:.4f}')
    print(f'Overall Recall   : {overall_recall:.4f}')

=== Hybrid Overall Metrics ===
TP: 927, FP: 81, FN: 101
Overall Precision: 0.9196
Overall Recall   : 0.9018


In [10]:
# Hybrid predictions for all test images and save outputs (clear visualization)
import os
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageFont

# Output directory
PRED_SAVE_DIR = ARTIFACTS_DIR / "hybrid_test_predictions"
os.makedirs(PRED_SAVE_DIR, exist_ok=True)

# Ensure required variables/models exist
required = ['TEST_IMAGES', 'yolo', 'CLASS_NAMES', 'prototypes', 'thresholds',
            'EVAL_DEVICE', 'YOLO_MIN_CONF', 'ALPHA', 'BETA', 'FUSION_THRESHOLD', 'extract_feature']
missing = [k for k in required if k not in globals()]
if missing:
    print('Run Cell 14 first. Missing:', ', '.join(missing))
else:
    valid_ext = {'.jpg', '.jpeg', '.png', '.bmp'}
    image_files = [f for f in os.listdir(TEST_IMAGES) if os.path.splitext(f)[1].lower() in valid_ext]
    image_files.sort()

    # Distinct colors per class (high contrast)
    class_colors = {
        0: (255, 64, 64),   # Gun
        1: (64, 200, 255),  # Knife
        2: (80, 255, 80),   # Pliers
        3: (255, 200, 64),  # Scissors
        4: (220, 120, 255)  # Wrench
    }

    saved_count = 0
    det_count_total = 0

    print(f'Running hybrid prediction on {len(image_files)} test images...')

    for idx, img_name in enumerate(image_files, start=1):
        img_path = os.path.join(TEST_IMAGES, img_name)
        img = Image.open(img_path).convert('RGB')
        w, h = img.size

        # Scaled line width and font size for readability on large/small images
        line_w = max(2, int(round(min(w, h) * 0.004)))
        font_size = max(14, int(round(min(w, h) * 0.028)))

        # Try TrueType first, fallback to default bitmap font
        try:
            font = ImageFont.truetype('arial.ttf', font_size)
        except Exception:
            font = ImageFont.load_default()

        # YOLO detections first
        result = yolo.predict(
            str(img_path),
            conf=YOLO_MIN_CONF,
            device=0 if EVAL_DEVICE == 'cuda' else 'cpu',
            verbose=False
        )[0]

        final_dets = []
        if result.boxes is not None and len(result.boxes) > 0:
            for box, cls_id, conf in zip(result.boxes.xyxy, result.boxes.cls, result.boxes.conf):
                cls_id = int(cls_id)
                if cls_id < 0 or cls_id >= len(CLASS_NAMES):
                    continue

                x1, y1, x2, y2 = map(int, box.tolist())
                x1 = max(0, min(w - 1, x1)); y1 = max(0, min(h - 1, y1))
                x2 = max(0, min(w, x2));     y2 = max(0, min(h, y2))
                if x2 <= x1 or y2 <= y1:
                    continue

                # SSL verification
                crop = img.crop((x1, y1, x2, y2))
                feat = extract_feature(crop)
                sims = F.cosine_similarity(feat, prototypes[CLASS_NAMES[cls_id]])
                ssl_score = float((sims.max().item() + 1.0) / 2.0)

                if ssl_score < thresholds[CLASS_NAMES[cls_id]]:
                    continue

                final_score = ALPHA * float(conf) + BETA * ssl_score
                if final_score < FUSION_THRESHOLD:
                    continue

                final_dets.append((x1, y1, x2, y2, cls_id, final_score))

        # Draw detections
        canvas = img.copy()
        draw = ImageDraw.Draw(canvas)

        for x1, y1, x2, y2, cls_id, score in final_dets:
            color = class_colors.get(cls_id, (255, 0, 0))

            # Thick box outline
            draw.rectangle([x1, y1, x2, y2], outline=color, width=line_w)

            # Label text and background box
            label = f"{CLASS_NAMES[cls_id]} {score:.2f}"
            text_bbox = draw.textbbox((0, 0), label, font=font)
            text_w = text_bbox[2] - text_bbox[0]
            text_h = text_bbox[3] - text_bbox[1]
            pad = max(3, line_w)

            tx1 = x1
            ty1 = y1 - text_h - (2 * pad)
            if ty1 < 0:
                ty1 = y1 + 1
            tx2 = min(w - 1, tx1 + text_w + (2 * pad))
            ty2 = min(h - 1, ty1 + text_h + (2 * pad))

            # black background + colored border for readability
            draw.rectangle([tx1, ty1, tx2, ty2], fill=(0, 0, 0), outline=color, width=max(1, line_w // 2))
            draw.text((tx1 + pad, ty1 + pad), label, fill=(255, 255, 255), font=font)

        out_path = PRED_SAVE_DIR / img_name
        canvas.save(out_path)

        saved_count += 1
        det_count_total += len(final_dets)

        if idx % 100 == 0:
            print(f'  processed {idx}/{len(image_files)} images')

    avg_det = det_count_total / max(saved_count, 1)
    print('\n=== Hybrid prediction export complete ===')
    print(f'Saved images: {saved_count}')
    print(f'Total detections kept: {det_count_total}')
    print(f'Average detections/image: {avg_det:.2f}')
    print(f'Output directory: {PRED_SAVE_DIR}')

Running hybrid prediction on 541 test images...
  processed 100/541 images
  processed 200/541 images
  processed 300/541 images
  processed 400/541 images
  processed 500/541 images

=== Hybrid prediction export complete ===
Saved images: 541
Total detections kept: 1008
Average detections/image: 1.86
Output directory: d:\xray2\xray2\artifacts\hybrid_test_predictions
